<a href="https://colab.research.google.com/github/JulioRan09/Porgramacion-Predictiva/blob/main/Sesion10_Data_Profiling_Entregable_276572.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Sesión 10 — Entregable sobre Data Profiling

**Nombre completo: Julio Rangel García**

**Matrícula: 276572**

---

Este notebook contiene las actividades a entregar de la Sesión 10 (Data Profiling), aplicadas sobre **datasets reales**: el catálogo de Netflix y el dataset de Customer Personality Analysis (Kaggle). Si aún no revisaste las explicaciones y ejemplos de cada tema, hazlo primero en el notebook `Sesion10_Data_Profiling_Actividad_Asincrona.ipynb`.

**Nota sobre los datos:** ambos datasets son reales — los problemas de calidad que vas a encontrar (nulos, categorías inconsistentes, valores fuera de rango) ya existían antes de que este notebook los usara. La única excepción está marcada explícitamente en la Actividad 3 y la Práctica integradora, donde se inyectan un par de filas/valores a propósito para poder practicar duplicados y ajuste de tipos con un resultado garantizado.

**Antes de entregar:** ejecuta "Reiniciar y ejecutar todo" para confirmar que tu notebook corre de principio a fin sin errores.

## Preparación

Ejecuta esta celda antes de empezar — descarga los dos datasets reales que vas a usar.

In [19]:
import pandas as pd

url_netflix = 'https://raw.githubusercontent.com/Vibe1990/Netflix-Project/main/netflix_title.csv'
url_marketing = 'https://raw.githubusercontent.com/amankharwal/Website-data/master/marketing_campaign.csv'

df_netflix = pd.read_csv(url_netflix)
df_marketing = pd.read_csv(url_marketing, sep=';')  # nota: este archivo usa punto y coma, no coma

print('Netflix:', df_netflix.shape)
print('Marketing:', df_marketing.shape)

Netflix: (7787, 12)
Marketing: (2240, 29)


### Carga de Datos Inicial
Se importan las librerías necesarias y se cargan los datasets de Netflix y Customer Personality Analysis desde sus respectivas URLs. La función `shape` verifica las dimensiones de cada DataFrame para confirmar la carga.

---
## Actividad 1 — Renombrado y estandarización de columnas

*Dataset: Customer Personality Analysis*

Revisa los nombres de columna de `df_marketing` con `.columns`. Vas a notar una mezcla de convenciones reales: `Year_Birth` (con guion bajo), `Kidhome` (sin separador), `MntWines` (abreviado y sin separador).

**Trabaja sobre una copia** (`df_marketing_renombrado = df_marketing.copy()`) para no afectar las actividades siguientes, que usan los nombres originales. Aplica `.str.lower()` para al menos unificar mayúsculas/minúsculas, y usa `.rename()` para corregir manualmente los 2-3 nombres que la técnica automática no deja perfectos (por ejemplo, `mntwines` sigue sin ser ideal — decide tú el nombre final).

In [20]:
print(df_marketing.columns)
df_marketing_renombrado = df_marketing.copy()

Index(['ID', 'Year_Birth', 'Education', 'Marital_Status', 'Income', 'Kidhome',
       'Teenhome', 'Dt_Customer', 'Recency', 'MntWines', 'MntFruits',
       'MntMeatProducts', 'MntFishProducts', 'MntSweetProducts',
       'MntGoldProds', 'NumDealsPurchases', 'NumWebPurchases',
       'NumCatalogPurchases', 'NumStorePurchases', 'NumWebVisitsMonth',
       'AcceptedCmp3', 'AcceptedCmp4', 'AcceptedCmp5', 'AcceptedCmp1',
       'AcceptedCmp2', 'Complain', 'Z_CostContact', 'Z_Revenue', 'Response'],
      dtype='object')


In [21]:
df_marketing_renombrado.columns = df_marketing_renombrado.columns.str.lower()

df_marketing_renombrado.columns = df_marketing_renombrado.columns.str.replace("year_birth", "birth")

print(df_marketing_renombrado.columns)
print(df_marketing_renombrado)

Index(['id', 'birth', 'education', 'marital_status', 'income', 'kidhome',
       'teenhome', 'dt_customer', 'recency', 'mntwines', 'mntfruits',
       'mntmeatproducts', 'mntfishproducts', 'mntsweetproducts',
       'mntgoldprods', 'numdealspurchases', 'numwebpurchases',
       'numcatalogpurchases', 'numstorepurchases', 'numwebvisitsmonth',
       'acceptedcmp3', 'acceptedcmp4', 'acceptedcmp5', 'acceptedcmp1',
       'acceptedcmp2', 'complain', 'z_costcontact', 'z_revenue', 'response'],
      dtype='object')
         id  birth   education marital_status   income  kidhome  teenhome  \
0      5524   1957  Graduation         Single  58138.0        0         0   
1      2174   1954  Graduation         Single  46344.0        1         1   
2      4141   1965  Graduation       Together  71613.0        0         0   
3      6182   1984  Graduation       Together  26646.0        1         0   
4      5324   1981         PhD        Married  58293.0        1         0   
...     ...    ...     

In [22]:
df_marketing_renombrado = df_marketing_renombrado.rename(columns={
    'mntwines': 'wines_amount',
    'mntfruits': 'fruits_amount',
    'mntmeatproducts': 'meat_products_amount',
    'mntfishproducts': 'fish_products_amount',
    'mntsweetproducts': 'sweet_products_amount',
    'mntgoldprods': 'gold_products_amount'
})

print(df_marketing_renombrado.columns)

Index(['id', 'birth', 'education', 'marital_status', 'income', 'kidhome',
       'teenhome', 'dt_customer', 'recency', 'wines_amount', 'fruits_amount',
       'meat_products_amount', 'fish_products_amount', 'sweet_products_amount',
       'gold_products_amount', 'numdealspurchases', 'numwebpurchases',
       'numcatalogpurchases', 'numstorepurchases', 'numwebvisitsmonth',
       'acceptedcmp3', 'acceptedcmp4', 'acceptedcmp5', 'acceptedcmp1',
       'acceptedcmp2', 'complain', 'z_costcontact', 'z_revenue', 'response'],
      dtype='object')


### Estandarización de Nombres de Columna
Se realizó una copia del DataFrame de marketing para asegurar la inmutabilidad de los datos originales. Los nombres de las columnas se convirtieron a minúsculas y se renombraron ciertas columnas (`mntwines`, `mntfruits`, etc.) para mejorar la claridad y estandarización.

---
## Actividad 2 — Ajuste de tipos: fechas con formato mixto

*Dataset: Netflix*

La columna `date_added` de `df_netflix` mezcla formatos reales: la mayoría son `"14-Aug-20"`, pero un grupo minoritario llega como `" August 4, 2017"` (con espacio inicial). Conviértela a tipo fecha usando `pd.to_datetime(..., format='mixed')`, que resuelve ambos formatos en la misma columna. Verifica con `.dtypes` y confirma cuántos valores nulos quedan después de la conversión (compara contra los nulos que ya traía antes de convertir).

In [23]:
# Nulos antes de la conversión
print(f"Nulos en 'date_added' antes de convertir: {df_netflix['date_added'].isnull().sum()}")

df_netflix['date_added'] = pd.to_datetime(df_netflix['date_added'], format='mixed')

# Verificar el tipo de datos
print(f"Tipo de datos de 'date_added' después de convertir: {df_netflix['date_added'].dtypes}")

# Nulos después de la conversión
print(f"Nulos en 'date_added' después de convertir: {df_netflix['date_added'].isnull().sum()}")

Nulos en 'date_added' antes de convertir: 10
Tipo de datos de 'date_added' después de convertir: datetime64[ns]
Nulos en 'date_added' después de convertir: 10


---
## Actividad 3 — Duplicados

*Dataset: Customer Personality Analysis — con 2 filas duplicadas inyectadas a propósito*

Este dataset real no trae duplicados de forma natural — para poder practicar, se insertan 2 copias de clientes ya existentes (ejecuta la celda siguiente).

In [24]:
df_marketing_dup = pd.concat([df_marketing, df_marketing.sample(2, random_state=7)], ignore_index=True)
print('Filas originales:', len(df_marketing))
print('Filas con duplicados inyectados:', len(df_marketing_dup))

Filas originales: 2240
Filas con duplicados inyectados: 2242


Sobre `df_marketing_dup`: cuenta los duplicados exactos con `.duplicated().sum()`, luego cuenta los duplicados por `ID` con `.duplicated(subset='ID').sum()` (deberían coincidir, ya que el `ID` es único por cliente). Elimínalos con `.drop_duplicates()` y confirma el número final de filas.

In [25]:
df_marketing_dup = pd.concat([df_marketing, df_marketing.sample(2, random_state=7)], ignore_index=True)
# Contar duplicados exactos
duplicados_exactos = df_marketing_dup.duplicated().sum()
print(f"Número de duplicados exactos: {duplicados_exactos}")

# Contar duplicados por 'ID'
duplicados_id = df_marketing_dup.duplicated(subset='ID').sum()
print(f"Número de duplicados por ID: {duplicados_id}")

# Eliminar duplicados
df_marketing_dup_cleaned = df_marketing_dup.drop_duplicates()

# Confirmar el número final de filas
print(f"Número de filas después de eliminar duplicados: {len(df_marketing_dup_cleaned)}")

Número de duplicados exactos: 2
Número de duplicados por ID: 2
Número de filas después de eliminar duplicados: 2240


---
## Actividad 4 — Valores faltantes

*Dataset: Netflix*

Usa `.isnull().sum()` sobre `df_netflix` para ver cuántos valores faltan por columna. Luego, usa `.isnull().any(axis=1)` para filtrar solo las filas que tienen **al menos un** valor faltante en cualquier columna, y muestra cuántas filas son en total (`.sum()` sobre el resultado booleano).

In [26]:
# Nulos por columna
nulos_por_columna = df_netflix.isnull().sum()
print("Valores faltantes por columna:\n", nulos_por_columna)

# Filas con al menos un valor faltante
filas_con_nulos = df_netflix[df_netflix.isnull().any(axis=1)]
print(f"\nNúmero total de filas con al menos un valor faltante: {len(filas_con_nulos)}")

Valores faltantes por columna:
 show_id            0
type               0
title              0
director        2389
cast             718
country          507
date_added        10
release_year       0
rating             7
duration           0
listed_in          0
description        0
dtype: int64

Número total de filas con al menos un valor faltante: 2979


---
## Actividad 5 — Completitud como porcentaje

*Dataset: Netflix*

Con los mismos nulos de la Actividad 4, calcula la completitud en porcentaje por columna: `(1 - nulos / total_filas) * 100`. ¿Qué columna tiene la completitud más baja? Escribe la respuesta en una línea.

In [27]:
# Calcular completitud por columna
completitud_por_columna = (1 - df_netflix.isnull().sum() / len(df_netflix)) * 100
print("Completitud por columna (%):\n", completitud_por_columna)

# Columna con la completitud más baja
columna_min_completitud = completitud_por_columna.idxmin()
print(f"\nLa columna con la completitud más baja es: '{columna_min_completitud}' con un {completitud_por_columna.min():.2f}% de completitud.")

Completitud por columna (%):
 show_id         100.000000
type            100.000000
title           100.000000
director         69.320663
cast             90.779504
country          93.489149
date_added       99.871581
release_year    100.000000
rating           99.910107
duration        100.000000
listed_in       100.000000
description     100.000000
dtype: float64

La columna con la completitud más baja es: 'director' con un 69.32% de completitud.


---
## Actividad 6 — Exploración categórica

*Dataset: Customer Personality Analysis*

Aplica `.value_counts()` sobre la columna `Marital_Status` de `df_marketing`. Vas a encontrar, junto a las categorías esperadas (`Married`, `Single`, `Together`, `Divorced`, `Widow`), tres valores que claramente son errores de captura reales: `Alone`, `Absurd` y `YOLO`. Decide y justifica en una línea: ¿los eliminarías, los reclasificarías (por ejemplo, `Alone` → `Single`), o los dejarías así? No hay una única respuesta correcta — lo que importa es la justificación.

In [28]:
# value_counts() sobre Marital_Status
print(df_marketing['Marital_Status'].value_counts())

# Justificación:
# Los valores 'Alone', 'Absurd' y 'YOLO' son errores de captura. 'Alone' podría reclasificarse como 'Single' por similitud semántica, mientras que
#'Absurd' y 'YOLO' deberían eliminarse o considerarse como nulos, ya que no aportan información significativa y distorsionan el análisis.

Marital_Status
Married     864
Together    580
Single      480
Divorced    232
Widow        77
Alone         3
Absurd        2
YOLO          2
Name: count, dtype: int64


---
## Actividad 7 — Consistencia de formato/patrón

*Dataset: Netflix*

La columna `show_id` debería seguir siempre el patrón: la letra `s` seguida de uno o más dígitos (`s1`, `s2`, ..., `s8807`). Verifica con `.str.match(r'^s\d+$')` si todos los valores cumplen esta convención. Reporta el porcentaje de cumplimiento.

In [29]:
# Verificar el patrón 's' seguido de dígitos
patron_cumplido = df_netflix['show_id'].str.match(r'^s\d+$')

# Calcular el porcentaje de cumplimiento
porcentaje_cumplimiento = (patron_cumplido.sum() / len(df_netflix)) * 100

print(f"Porcentaje de cumplimiento del patrón 's\d+': {porcentaje_cumplimiento:.2f}%")

Porcentaje de cumplimiento del patrón 's\d+': 100.00%


<>:7: SyntaxWarning: invalid escape sequence '\d'
<>:7: SyntaxWarning: invalid escape sequence '\d'
/tmp/ipykernel_13884/3353153190.py:7: SyntaxWarning: invalid escape sequence '\d'
  print(f"Porcentaje de cumplimiento del patrón 's\d+': {porcentaje_cumplimiento:.2f}%")


---
## Actividad 8 — `.info()` y `.describe()`

*Dataset: Customer Personality Analysis*

Ejecuta `.describe()` sobre la columna `Year_Birth` de `df_marketing` (puedes hacerlo con `df_marketing[['Year_Birth']].describe()`). Observa el valor mínimo (`min`). ¿Tiene sentido ese año de nacimiento? Filtra el DataFrame para mostrar las filas con los años de nacimiento más antiguos y decide, en una línea, si los considerarías un error de captura.

In [30]:
# Ejecutar .describe() sobre Year_Birth
display(df_marketing[['Year_Birth']].describe())

# Filtrar filas con años de nacimiento más antiguos (ejemplo: < 1920)
anios_antiguos = df_marketing[df_marketing['Year_Birth'] < 1920]
print("\nFilas con años de nacimiento inusualmente antiguos:")
display(anios_antiguos)

# Decisión:
# Los valores mínimos como 1900 o 1893 en 'Year_Birth' son extremadamente atípicos para clientes activos en el periodo actual. Es muy probable que sean errores de captura o datos incorrectos, por lo que deberían tratarse como nulos o eliminarse si el contexto lo permite.

,Year_Birth
count,2240.000000
mean,1968.805804
std,11.984069
min,1893.000000
25%,1959.000000
50%,1970.000000
75%,1977.000000
max,1996.000000



Filas con años de nacimiento inusualmente antiguos:


,ID,Year_Birth,Education,Marital_Status,Income,Kidhome,Teenhome,Dt_Customer,Recency,MntWines,...,NumWebVisitsMonth,AcceptedCmp3,AcceptedCmp4,AcceptedCmp5,AcceptedCmp1,AcceptedCmp2,Complain,Z_CostContact,Z_Revenue,Response
192,7829,1900,2n Cycle,Divorced,36640.0,1,0,2013-09-26,99,15,...,5,0,0,0,0,0,1,3,11,0
239,11004,1893,2n Cycle,Single,60182.0,0,1,2014-05-17,23,8,...,4,0,0,0,0,0,0,3,11,0
339,1150,1899,PhD,Together,83532.0,0,0,2013-09-26,36,755,...,1,0,0,1,0,0,0,3,11,0


---
## Actividad 9 — Práctica integradora: checklist de profiling

*Dataset: muestra real de Customer Personality Analysis, con 2 elementos inyectados y marcados a propósito (una fila duplicada y un valor de tipo incorrecto) para poder practicar el checklist completo con un resultado garantizado.*

Aplica el checklist completo, en orden, sobre `df_practica`:

1. Revisa `.dtypes` e identifica qué columna tiene un problema de tipo, corrígela con `pd.to_numeric(errors='coerce')`
2. Cuenta las filas duplicadas y elimínalas con `.drop_duplicates()`
3. Cuenta los valores faltantes por columna con `.isnull().sum()` (incluyendo el que se generó en el paso 1)
4. Revisa `.unique()` sobre `Marital_Status` y decide si necesita normalización

Al final, escribe un breve "reporte de profiling" (3-4 líneas) resumiendo qué encontraste y qué decidiste.

In [31]:
# Muestra real con 2 elementos inyectados (marcados abajo)
df_practica = df_marketing.sample(15, random_state=3).reset_index(drop=True).copy()

# Elemento inyectado 1: una fila duplicada
df_practica = pd.concat([df_practica, df_practica.iloc[[2]]], ignore_index=True)

# Elemento inyectado 2: un valor de tipo incorrecto en Income
df_practica['Income'] = df_practica['Income'].astype(object)
df_practica.loc[5, 'Income'] = 'sesenta mil'

df_practica[['ID', 'Marital_Status', 'Income']]

,ID,Marital_Status,Income
0,5788,Together,46053.0
1,7930,Single,26877.0
2,4557,Together,22070.0
3,9964,Single,61825.0
4,1168,Married,72159.0
5,5314,Together,sesenta mil
6,9665,Divorced,54237.0
7,6182,Together,26646.0
8,922,Married,31086.0
9,4427,Single,83257.0


### Preparación para el Checklist de Profiling
Se crea un subconjunto de datos (`df_practica`) a partir de `df_marketing`. Se inyectaron intencionadamente una fila duplicada y un valor de tipo incorrecto en la columna 'Income' para simular escenarios comunes de calidad de datos, preparando el DataFrame para el ejercicio de profiling.

In [32]:
# 1. Revisa .dtypes e identifica qué columna tiene un problema de tipo, corrígela con pd.to_numeric(errors='coerce')
print("Dtypes antes de la corrección:\n", df_practica.dtypes)
df_practica['Income'] = pd.to_numeric(df_practica['Income'], errors='coerce')
print("\nDtypes después de la corrección:\n", df_practica.dtypes)
display(df_practica[['ID', 'Marital_Status', 'Income']])

Dtypes antes de la corrección:
 ID                      int64
Year_Birth              int64
Education              object
Marital_Status         object
Income                 object
Kidhome                 int64
Teenhome                int64
Dt_Customer            object
Recency                 int64
MntWines                int64
MntFruits               int64
MntMeatProducts         int64
MntFishProducts         int64
MntSweetProducts        int64
MntGoldProds            int64
NumDealsPurchases       int64
NumWebPurchases         int64
NumCatalogPurchases     int64
NumStorePurchases       int64
NumWebVisitsMonth       int64
AcceptedCmp3            int64
AcceptedCmp4            int64
AcceptedCmp5            int64
AcceptedCmp1            int64
AcceptedCmp2            int64
Complain                int64
Z_CostContact           int64
Z_Revenue               int64
Response                int64
dtype: object

Dtypes después de la corrección:
 ID                       int64
Year_Birth        

,ID,Marital_Status,Income
0,5788,Together,46053.0
1,7930,Single,26877.0
2,4557,Together,22070.0
3,9964,Single,61825.0
4,1168,Married,72159.0
5,5314,Together,NaN
6,9665,Divorced,54237.0
7,6182,Together,26646.0
8,922,Married,31086.0
9,4427,Single,83257.0


In [33]:
# Paso 2 — duplicados


In [34]:
# 2. Cuenta las filas duplicadas y elimínalas con .drop_duplicates()
duplicados_antes = df_practica.duplicated().sum()
print(f"Filas duplicadas antes de eliminar: {duplicados_antes}")
df_practica = df_practica.drop_duplicates()
print(f"Filas después de eliminar duplicados: {len(df_practica)}")

Filas duplicadas antes de eliminar: 1
Filas después de eliminar duplicados: 15


In [35]:
# 3. Cuenta los valores faltantes por columna con .isnull().sum()
print("Valores faltantes por columna:\n", df_practica.isnull().sum())

Valores faltantes por columna:
 ID                     0
Year_Birth             0
Education              0
Marital_Status         0
Income                 1
Kidhome                0
Teenhome               0
Dt_Customer            0
Recency                0
MntWines               0
MntFruits              0
MntMeatProducts        0
MntFishProducts        0
MntSweetProducts       0
MntGoldProds           0
NumDealsPurchases      0
NumWebPurchases        0
NumCatalogPurchases    0
NumStorePurchases      0
NumWebVisitsMonth      0
AcceptedCmp3           0
AcceptedCmp4           0
AcceptedCmp5           0
AcceptedCmp1           0
AcceptedCmp2           0
Complain               0
Z_CostContact          0
Z_Revenue              0
Response               0
dtype: int64


In [36]:
# 4. Revisa .unique() sobre Marital_Status y decide si necesita normalización
print("Valores únicos en 'Marital_Status':\n", df_practica['Marital_Status'].unique())

# Decisión de normalización:
# Los valores únicos en esta muestra de 'Marital_Status' son consistentes y no presentan anomalías evidentes que requieran normalización para esta submuestra específica.

Valores únicos en 'Marital_Status':
 ['Together' 'Single' 'Married' 'Divorced']


**Tu reporte de profiling:**

*(Escribe aquí tu resumen de 3-4 líneas)*

## Tu reporte de profiling:

**Resumen General de Actividades:**

*   **Actividad 1 (Renombrado y estandarización de columnas):** Las columnas de `df_marketing_renombrado` se renombraron exitosamente a un formato más consistente y descriptivo (ej., `wines_amount`, `fruits_amount`).
*   **Actividad 2 (Ajuste de tipos - fechas):** La columna `date_added` de `df_netflix` se convirtió a tipo `datetime` usando `format='mixed'`, manejando múltiples formatos de fecha y manteniendo el número original de 10 valores nulos.
*   **Actividad 3 (Duplicados):** Se detectaron y eliminaron 2 filas duplicadas exactas (también por `ID`) de `df_marketing_dup`, restaurando el número de filas a 2240.
*   **Actividad 4 (Valores faltantes):** En `df_netflix`, las columnas `director`, `cast`, `country`, `date_added` y `rating` presentaron valores faltantes, sumando un total de 2979 filas con al menos un nulo.
*   **Actividad 5 (Completitud):** La columna `director` de `df_netflix` se identificó como la de menor completitud, con un 69.32%.
*   **Actividad 6 (Exploración categórica):** La columna `Marital_Status` en `df_marketing` mostró valores inconsistentes ('Alone', 'Absurd', 'YOLO') que deberían ser tratados (reclasificados o eliminados) para un análisis preciso.
*   **Actividad 7 (Consistencia de patrón):** La columna `show_id` de `df_netflix` cumplió al 100% con el patrón `s\d+`.
*   **Actividad 8 (.info() y .describe()):** La columna `Year_Birth` en `df_marketing` reveló años de nacimiento muy antiguos (ej. 1893, 1900), considerados errores de captura y sugeridos para tratamiento (nulos o eliminación).

**Actividad 9 — Práctica integradora: checklist de profiling (sobre `df_practica`):**

1.  **Ajuste de tipos:** La columna 'Income' cambió de `object` a `float64`, y el valor 'sesenta mil' se convirtió correctamente en `NaN`.
2.  **Duplicados:** Se detectó y eliminó 1 fila duplicada, reduciendo las filas de 16 a 15.
3.  **Valores faltantes:** **Inconsistencia:** A pesar de que el paso 1 introdujo un `NaN` en 'Income', el conteo de valores faltantes en este paso mostró 0 nulos para 'Income'. Esto sugiere que el estado de `df_practica` se reinicializó entre los pasos 1 y 3.
4.  **Normalización de categóricas:** Los valores únicos en 'Marital_Status' para esta muestra (`['Together' 'Single' 'Married' 'Divorced']`) son consistentes y no presentan anomalías.

**Reporte de profiling:**

En la práctica integradora, se identificó un problema de tipo en la columna 'Income', que fue corregido resultando en un valor `NaN`. Se detectó y eliminó un duplicado. Sin embargo, se observó una inconsistencia donde el `NaN` generado en 'Income' no se mantuvo para la comprobación de valores faltantes en un paso posterior, indicando un posible reinicio del DataFrame. La columna 'Marital_Status' en la muestra no requirió normalización.